# 01 · Ingesta y caracterización — SNIIV

Carga los extractos del cubo SNIIV (acciones y monto) a formato *tidy* usando el
módulo `src/ingesta_sniiv.py`, verifica los totales conocidos, caracteriza el
resultado y guarda una salida intermedia en `data/interim/`.

Las trampas del dato (qué significan `valor_vivienda`, `monto` y "No disponible")
están documentadas en [`docs/decisiones.md`](../docs/decisiones.md).

## Configuración

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Raíz del repo (el notebook corre desde notebooks/)
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.append(str(REPO / "src"))

from ingesta_sniiv import cargar_sniiv

RAW = REPO / "data" / "raw"
INTERIM = REPO / "data" / "interim"
INTERIM.mkdir(exist_ok=True)

## Carga

Se localizan los dos extractos por patrón de nombre (robusto a la fecha del archivo).

In [2]:
ruta_acciones = next(RAW.glob("infonavit_acciones-segmento_merida_*.xls"))
ruta_monto = next(RAW.glob("infonavit_monto-segmento_merida_*.xls"))

df = cargar_sniiv(ruta_acciones, ruta_monto)
df.head()

,anio,segmento,acciones,monto
0,2015,Económica,306,5.464735e+07
1,2015,Media,697,2.812566e+08
2,2015,No disponible,3536,0.000000e+00
3,2015,Popular,3426,8.865023e+08
4,2015,Residencial,74,3.108829e+07


## Forma y tipos

In [3]:
print("Forma:", df.shape)  # (77, 4) = 11 años × 7 segmentos
df.dtypes

Forma: (77, 4)


anio          int64
segmento        str
acciones      Int64
monto       float64
dtype: object

## Totales y chequeos de cordura

Se valida el pipeline contra los totales conocidos del cubo (Mérida 2015–2025):
**69,448** acciones y **$24,493,507,058.80** de monto.

In [4]:
total_acciones = int(df["acciones"].sum())
total_monto = float(df["monto"].sum())
print(f"Total acciones: {total_acciones:,}")
print(f"Total monto:    ${total_monto:,.2f}")

assert total_acciones == 69_448, total_acciones
assert round(total_monto, 2) == 24_493_507_058.80, total_monto
print("Chequeos de cordura: OK")

Total acciones: 69,448
Total monto:    $24,493,507,058.80
Chequeos de cordura: OK


## Distribución por segmento

`monto_prom_credito` = monto / acciones es el **crédito promedio**, no el valor de
la vivienda (ver `docs/decisiones.md`).

In [5]:
resumen = (
    df.groupby("segmento")
    .agg(acciones=("acciones", "sum"), monto=("monto", "sum"))
    .assign(
        part_acciones=lambda x: (x["acciones"] / x["acciones"].sum() * 100).round(1),
        monto_prom_credito=lambda x: (x["monto"] / x["acciones"]).round(0),
    )
    .sort_values("acciones", ascending=False)
)
resumen

,acciones,monto,part_acciones,monto_prom_credito
segmento,,,,
No disponible,19808,8.435948e+06,28.5,426.0
Popular,19120,6.382306e+09,27.5,333803.0
Tradicional,17521,8.381496e+09,25.2,478369.0
Media,10493,8.166900e+09,15.1,778319.0
Residencial,1432,1.314475e+09,2.1,917930.0
Económica,917,1.805680e+08,1.3,196912.0
Residencial plus,157,5.932564e+07,0.2,377870.0


## "No disponible"

Casi un tercio de los créditos, con monto ≈ 0. Se reporta como dato, pero se
excluye del análisis de valor.

In [6]:
mask = df["segmento"] == "No disponible"
acc_nd = int(df.loc[mask, "acciones"].sum())
print(f"No disponible — acciones: {acc_nd:,} "
      f"({acc_nd / total_acciones * 100:.1f}% del total)")
print(f"No disponible — monto:    ${df.loc[mask, 'monto'].sum():,.2f}")

No disponible — acciones: 19,808 (28.5% del total)
No disponible — monto:    $8,435,947.99


## Salida intermedia

In [7]:
salida = INTERIM / "sniiv_merida_tidy.parquet"
df.to_parquet(salida, index=False)
print("Guardado:", salida.relative_to(REPO))

Guardado: data/interim/sniiv_merida_tidy.parquet
